In [ ]:
import asyncio
import random
import datetime
import redis.asyncio as redis
import nest_asyncio

nest_asyncio.apply()

num_test_streams = 3
pub_freq = 1
stream_max_len = 100

async def publish_test_data_for_stream(stream_index, redis_client):
    last_price = 100.0  # Starting price
    stream_key = f"test_{stream_index}"  # Using test_1, test_2, etc.

    while True:
        # Simulate large swings by adding more volatility
        change = random.uniform(-5, 5)  # Increased fluctuation range
        last_price = max(10, last_price + change)  # Keep price above zero

        # Force RSI boundary conditions sometimes
        if random.random() < 0.1:  
            last_price *= random.choice([0.85, 1.15])  # Big jumps 15% up or down

        # Create fake OHLC data
        data = {
            "symbol": "TEST",
            "timestamp": datetime.datetime.now(datetime.UTC).isoformat(),
            "open": round(last_price - random.uniform(0.5, 2), 2),
            "high": round(last_price + random.uniform(0.5, 2), 2),
            "low": round(last_price - random.uniform(1, 3), 2),
            "close": round(last_price, 2),
            "volume": random.randint(100, 1000),
            "trade_count": random.randint(10, 50),
            "vwap": round(last_price + random.uniform(-1, 1), 2),
        }

        # Push data to Redis stream
        await redis_client.xadd(stream_key, data, maxlen=stream_max_len)
        print(f"Pushed to {stream_key}: {data}")

        await asyncio.sleep(pub_freq)  # Adjust frequency if needed

async def publish_test_data(num_streams=1):
    redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

    # Create a list of tasks to run multiple streams concurrently
    tasks = []
    for stream_index in range(1, num_streams + 1):
        task = asyncio.create_task(publish_test_data_for_stream(stream_index, redis_client))
        tasks.append(task)

    # Run all the tasks concurrently
    await asyncio.gather(*tasks)


await publish_test_data(num_streams=num_test_streams)

Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-03-22T13:59:09.925753+00:00', 'open': 96.6, 'high': 99.87, 'low': 96.37, 'close': 98.08, 'volume': 780, 'trade_count': 25, 'vwap': 97.85}
Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-03-22T13:59:09.925753+00:00', 'open': 101.48, 'high': 103.83, 'low': 100.14, 'close': 102.16, 'volume': 415, 'trade_count': 36, 'vwap': 102.36}
Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-03-22T13:59:09.925753+00:00', 'open': 94.84, 'high': 97.05, 'low': 93.66, 'close': 95.8, 'volume': 664, 'trade_count': 44, 'vwap': 96.2}
Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-03-22T13:59:10.929881+00:00', 'open': 102.47, 'high': 105.39, 'low': 102.65, 'close': 104.34, 'volume': 367, 'trade_count': 18, 'vwap': 104.11}
Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-03-22T13:59:10.929881+00:00', 'open': 92.7, 'high': 96.63, 'low': 92.64, 'close': 94.67, 'volume': 144, 'trade_count': 34, 'vwap': 94.18}
Pushed to test_1: {